# CodeDrift Arena — OpenEnv on Colab

Run the CodeDrift Arena OpenEnv server on Colab and drive it from an agent (WebSocket client).

**What this notebook does**
1. Clone the repo and install server deps
2. Boot `uvicorn server.app:app` in the background on port 8000
3. Hit `/health` and `/metadata` to confirm the server is up
4. Run `scripts/openenv_ws_demo.py` — one full `reset` → `step` episode over the `/ws` WebSocket
5. Drive an episode directly from a notebook cell (so you can plug your own agent logic in)
6. (Optional) Expose the server publicly via cloudflared so an external agent / trainer can connect

**Why WebSocket:** stateless `POST /reset` + `POST /step` each spin up a *new* env in openenv-core, so `step` cannot follow `reset` over plain HTTP. The `/ws` endpoint keeps a single `CodeDriftOpenEnvironment` alive for the connection.

## 1. Clone the repo and install server dependencies

The server stack is slim — no torch / unsloth / bitsandbytes. You do not need a GPU runtime just to run the environment.

In [ ]:
!git clone https://github.com/bansalbhunesh/codedrift-arena.git
%cd codedrift-arena
!pip install -q -r requirements-server.txt

## 2. Start the OpenEnv server in the background

Uvicorn runs as a child process and logs to `logs/uvicorn.out`. If `/health` does not respond in step 3, `!tail -80 logs/uvicorn.out` to see the startup error.

In [ ]:
import os, subprocess, time, signal, atexit

os.makedirs('logs', exist_ok=True)

# Kill any previous uvicorn in this runtime (safe to re-run the cell)
subprocess.run(['pkill', '-f', 'uvicorn server.app:app'], check=False)
time.sleep(1)

log_fh = open('logs/uvicorn.out', 'w')
server_proc = subprocess.Popen(
    ['uvicorn', 'server.app:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=log_fh, stderr=subprocess.STDOUT,
)

def _cleanup():
    try:
        server_proc.terminate()
        server_proc.wait(timeout=5)
    except Exception:
        try: server_proc.kill()
        except Exception: pass
atexit.register(_cleanup)

# Wait for readiness
import urllib.request, json
ready = False
for _ in range(30):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2) as r:
            if r.status == 200:
                ready = True
                break
    except Exception:
        time.sleep(1)
print('server pid:', server_proc.pid, 'ready:', ready)

## 3. Sanity-check the HTTP surface

In [ ]:
!curl -s http://127.0.0.1:8000/health
!echo
!curl -s http://127.0.0.1:8000/metadata | head -c 1500
!echo

If either curl fails, inspect the server log:

In [ ]:
!tail -80 logs/uvicorn.out

## 4. Run the built-in WebSocket demo

One `reset` → `step` episode with a hard-coded agent response. This is the canonical “agent talks to OpenEnv” flow.

In [ ]:
!python scripts/openenv_ws_demo.py --seed 7

## 5. Drive an episode directly (plug your own agent in here)

Same protocol as the demo script, but inline so you can replace the hard-coded review with your own agent's output — e.g. an LLM call that inspects the reset observation and emits a `VERDICT` / `ISSUES` / `REASON` response.

In [ ]:
import asyncio, json, websockets

WS_URL = 'ws://127.0.0.1:8000/ws'

def my_agent(reset_observation: dict) -> str:
    """Replace this with your real agent. Returns the review text the env will score."""
    # reset_observation['data']['observation'] has the PR context the env produced.
    return (
        'VERDICT: REQUEST_CHANGES\n'
        'ISSUES: stale API references in diff\n'
        'REASON: Placeholder agent response — swap in your model here.\n'
    )

async def run_one(seed: int = 42):
    async with websockets.connect(WS_URL) as ws:
        await ws.send(json.dumps({'type': 'reset', 'data': {'seed': seed}}))
        reset_payload = json.loads(await ws.recv())
        print('--- reset ---')
        print(json.dumps(reset_payload, indent=2)[:2000])

        review = my_agent(reset_payload)
        await ws.send(json.dumps({
            'type': 'step',
            'data': {'metadata': {'agent_response': review}},
        }))
        step_payload = json.loads(await ws.recv())
        print('--- step ---')
        print(json.dumps(step_payload, indent=2)[:2000])

        data = step_payload.get('data') or {}
        obs = (data.get('observation') or {})
        info = obs.get('scorer_info') or {}
        print('\nreward=', data.get('reward'),
              '\ndone=', data.get('done'),
              '\nverdict=', info.get('verdict'),
              '\noutcome=', info.get('episode_outcome'),
              '\nmetric_strip=', info.get('metric_strip'))

# In Colab, asyncio is already running — use await directly, not asyncio.run()
await run_one(seed=42)

## 6. (Optional) Expose the server publicly with cloudflared

Use this if you want a trainer or agent running **outside** this Colab runtime to connect. No signup required.

After this cell prints a `https://…trycloudflare.com` URL, your WebSocket endpoint is `wss://<that-host>/ws`.

In [ ]:
import subprocess, time, re, os
os.makedirs('logs', exist_ok=True)

if not os.path.exists('cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

subprocess.run(['pkill', '-f', 'cloudflared'], check=False)
tunnel_proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8000'],
    stdout=open('logs/tunnel.out', 'w'), stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(30):
    time.sleep(1)
    try:
        text = open('logs/tunnel.out').read()
        m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', text)
        if m:
            public_url = m.group(0)
            break
    except FileNotFoundError:
        pass

print('public HTTP:', public_url)
if public_url:
    print('public WS  :', public_url.replace('https://', 'wss://') + '/ws')
else:
    print('(no URL yet — tail logs/tunnel.out)')

## 7. Shut down (optional)

Colab reclaims the runtime on disconnect, but if you want to reset state without losing the notebook:

In [ ]:
!pkill -f 'uvicorn server.app:app' || true
!pkill -f cloudflared || true
print('stopped')